In [1]:
import sys
print(sys.executable)
import ultralytics
print(ultralytics.__version__)

D:\AI-Based Garbage Detection\venv\Scripts\python.exe
8.4.173


In [2]:
import glob, shutil, os, yaml
from collections import Counter

BASE = 'D:/AI-Based Garbage Detection'
DATA = BASE + '/dataset'
OUT = BASE + '/dataset_final'

cfg = yaml.safe_load(open(DATA + '/data.yaml'))
names = cfg['names']
NAMES = [names[i] for i in sorted(names)] if isinstance(names, dict) else list(names)
VAL = 'valid' if os.path.isdir(DATA + '/valid') else 'val'

tr = Counter()
for f in glob.glob(DATA + '/train/labels/*.txt'):
    for line in open(f):
        p = line.split()
        if len(p) >= 5:
            tr[int(p[0])] += 1

MIN_TRAIN_OBJECTS = 100
MERGE = {
    'Zip plastic bag': 'Plastic bag',
    'Paper bag': 'Paper',
    'Paper shavings': 'Paper',
    'Cellulose': 'Paper',
}
final_name = {c: MERGE.get(NAMES[c], NAMES[c]) for c in range(len(NAMES))}
merged_count = Counter()
for c in range(len(NAMES)):
    merged_count[final_name[c]] += tr[c]

keep_names = []
for n in dict.fromkeys(final_name.values()):
    if merged_count[n] >= MIN_TRAIN_OBJECTS:
        keep_names.append(n)

new_id = {n: i for i, n in enumerate(keep_names)}
old2new = {}
for c in range(len(NAMES)):
    if final_name[c] in new_id:
        old2new[c] = new_id[final_name[c]]

print(len(NAMES), 'classes ->', len(keep_names), 'classes')
print('Kept:', keep_names)

EXTS = ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']

def find_image(img_dir, stem):
    for e in EXTS:
        p = img_dir + '/' + stem + e
        if os.path.exists(p):
            return p
    return None

shutil.rmtree(OUT, ignore_errors=True)

for split in ['train', VAL, 'test']:
    lbl_dir = DATA + '/'

42 classes -> 11 classes
Kept: ['Aluminum can', 'Cardboard', 'Paper', 'Container for household chemicals', 'Glass bottle', 'Organic', 'Plastic bag', 'Plastic bottle', 'Plastic cup', 'Printing industry', 'Tin']


In [5]:
import glob, os

found = []
for pattern in ['C:/Users/*/Downloads/*.pt', 'D:/**/*.pt']:
    for p in glob.glob(pattern, recursive=True):
        if 'venv' in p:
            continue
        found.append((p, round(os.path.getsize(p) / 1e6, 1)))
for p, mb in sorted(found):
    print(mb, 'MB', p)

52.0 MB C:/Users\paria\Downloads\best.pt
0.0 MB D:/$RECYCLE.BIN\S-1-5-21-3470826629-177345985-2732710280-1001\$ILFAJZ7.pt
6.5 MB D:/$RECYCLE.BIN\S-1-5-21-3470826629-177345985-2732710280-1001\$RLFAJZ7.pt
6.2 MB D:/AI-Based Garbage Detection\runs\detect\garbage_9class_10\weights\best.pt
6.2 MB D:/AI-Based Garbage Detection\runs\detect\garbage_9class_10\weights\last.pt
6.2 MB D:/AI-Based Garbage Detection\runs\detect\garbage_detection-2\weights\best.pt
6.2 MB D:/AI-Based Garbage Detection\runs\detect\garbage_detection-2\weights\last.pt
6.2 MB D:/AI-Based Garbage Detection\runs\detect\train\weights\best.pt
6.2 MB D:/AI-Based Garbage Detection\runs\detect\train\weights\last.pt
6.5 MB D:/AI-Based Garbage Detection\yolov8n.pt


In [6]:
import shutil, os

shutil.copy(r'C:/Users/paria/Downloads/best.pt', BASE + '/models/best.pt')
print(os.listdir(BASE + '/models'))
print(round(os.path.getsize(BASE + '/models/best.pt') / 1e6, 1), 'MB')

['best.pt']
52.0 MB


In [8]:
import shutil, os
shutil.copy(r'C:/Users/paria/Downloads/best.pt', BASE + '/models/best.pt')
print(os.listdir(BASE + '/models'))

['best.pt']


In [10]:
import os, glob

print('dataset_final exists:', os.path.isdir(OUT))
if os.path.isdir(OUT):
    for split in os.listdir(OUT):
        p = OUT + '/' + split
        if os.path.isdir(p):
            print(split, 'images:', len(glob.glob(p + '/images/*')), ' labels:', len(glob.glob(p + '/labels/*')))
    print('Files in dataset_final:', os.listdir(OUT))
print('Original dataset folder exists:', os.path.isdir(DATA), os.listdir(DATA) if os.path.isdir(DATA) else '')

dataset_final exists: False
Original dataset folder exists: True ['data.yaml', 'README.dataset.txt', 'README.roboflow.txt', 'test', 'train', 'valid']


In [11]:
import glob, shutil, os, yaml
from collections import Counter

BASE = 'D:/AI-Based Garbage Detection'
DATA = BASE + '/dataset'
OUT = BASE + '/dataset_final'

cfg = yaml.safe_load(open(DATA + '/data.yaml'))
names = cfg['names']
NAMES = [names[i] for i in sorted(names)] if isinstance(names, dict) else list(names)
VAL = 'valid' if os.path.isdir(DATA + '/valid') else 'val'
print('Classes in original:', len(NAMES), '| validation folder:', VAL)

# Show what is inside each split folder
for split in ['train', VAL, 'test']:
    p = DATA + '/' + split
    print(split, '->', os.listdir(p) if os.path.isdir(p) else 'FOLDER MISSING')
    print('   images:', len(glob.glob(p + '/images/*')), ' labels:', len(glob.glob(p + '/labels/*.txt')))

tr = Counter()
for f in glob.glob(DATA + '/train/labels/*.txt'):
    for line in open(f):
        p = line.split()
        if len(p) >= 5:
            tr[int(p[0])] += 1
print('Total train objects counted:', sum(tr.values()))

MIN_TRAIN_OBJECTS = 100
MERGE = {
    'Zip plastic bag': 'Plastic bag',
    'Paper bag': 'Paper',
    'Paper shavings': 'Paper',
    'Cellulose': 'Paper',
}
final_name = {c: MERGE.get(NAMES[c], NAMES[c]) for c in range(len(NAMES))}
merged_count = Counter()
for c in range(len(NAMES)):
    merged_count[final_name[c]] += tr[c]

keep_names = []
for n in dict.fromkeys(final_name.values()):
    if merged_count[n] >= MIN_TRAIN_OBJECTS:
        keep_names.append(n)

new_id = {n: i for i, n in enumerate(keep_names)}
old2new = {}
for c in range(len(NAMES)):
    if final_name[c] in new_id:
        old2new[c] = new_id[final_name[c]]

print(len(NAMES), 'classes ->', len(keep_names), 'classes')
print('Kept:', keep_names)

EXTS = ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']

def find_image(img_dir, stem):
    for e in EXTS:
        p = img_dir + '/' + stem + e
        if os.path.exists(p):
            return p
    return None

shutil.rmtree(OUT, ignore_errors=True)
os.makedirs(OUT, exist_ok=True)

for split in ['train', VAL, 'test']:
    lbl_dir = DATA + '/' + split + '/labels'
    img_dir = DATA + '/' + split + '/images'
    if not os.path.isdir(lbl_dir):
        print('SKIPPED', split, '- no labels folder at', lbl_dir)
        continue
    os.makedirs(OUT + '/' + split + '/images', exist_ok=True)
    os.makedirs(OUT + '/' + split + '/labels', exist_ok=True)
    n = 0
    for f in os.listdir(lbl_dir):
        if not f.endswith('.txt'):
            continue
        new = []
        for line in open(lbl_dir + '/' + f):
            p = line.split()
            if len(p) == 5 and int(p[0]) in old2new:
                p[0] = str(old2new[int(p[0])])
                new.append(' '.join(p))
        if not new:
            continue
        img = find_image(img_dir, os.path.splitext(f)[0])
        if img is None:
            continue
        shutil.copy2(img, OUT + '/' + split + '/images/')
        with open(OUT + '/' + split + '/labels/' + f, 'w') as out_f:
            out_f.write('\n'.join(new) + '\n')
        n += 1
    print(split, n, 'images copied')

yaml.dump({'path': OUT, 'train': 'train/images', 'val': VAL + '/images', 'test': 'test/images',
           'nc': len(keep_names), 'names': keep_names},
          open(OUT + '/data.yaml', 'w'), sort_keys=False)
print('Done. Files in dataset_final:', os.listdir(OUT))

Classes in original: 42 | validation folder: valid
train -> ['images', 'labels']
   images: 3822  labels: 3822
valid -> ['images', 'labels']
   images: 1092  labels: 1092
test -> ['images', 'labels']
   images: 546  labels: 546
Total train objects counted: 5513
42 classes -> 11 classes
Kept: ['Aluminum can', 'Cardboard', 'Paper', 'Container for household chemicals', 'Glass bottle', 'Organic', 'Plastic bag', 'Plastic bottle', 'Plastic cup', 'Printing industry', 'Tin']
train 3728 images copied
valid 1064 images copied
test 533 images copied
Done. Files in dataset_final: ['data.yaml', 'test', 'train', 'valid']


In [12]:
import pandas as pd
from ultralytics import YOLO

BEST = BASE + '/models/best.pt'
model = YOLO(BEST)
m = model.val(data=OUT + '/data.yaml', imgsz=640, device='cpu', workers=0)

print('Overall mAP50 =', round(m.box.map50, 3))
rows = []
for k, c in enumerate(m.box.ap_class_index):
    rows.append({'class': model.names[int(c)], 'P': round(m.box.p[k], 2),
                 'R': round(m.box.r[k], 2), 'mAP50': round(m.box.ap50[k], 2)})
pd.DataFrame(rows).sort_values('mAP50')

Ultralytics 8.4.173  Python-3.12.7 torch-2.14.1+cpu CPU (13th Gen Intel Core i5-13420H)
Model summary (fused): 92 layers, 25,846,129 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 0.40.3 MB/s, size: 7.0 KB)
val: Scanning D:\AI-Based Garbage Detection\dataset_final\valid\labels... 1064 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1064/1064 110.1it/s 9.7s0.1s
val: New cache created: D:\AI-Based Garbage Detection\dataset_final\valid\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 67/67 19.1s/it 21:2221.5s
                   all       1064       1511      0.774      0.741       0.79      0.586
          Aluminum can        307        404      0.925      0.913      0.942      0.635
             Cardboard        107        133      0.765       0.71      0.783      0.582
                 Paper         33         33      0.926      0.759      0.835      0.701
Container for hous

,class,P,R,mAP50
3,Container for household chemicals,0.51,0.62,0.58
6,Plastic bag,0.67,0.53,0.63
7,Plastic bottle,0.69,0.66,0.69
10,Tin,0.64,0.65,0.71
5,Organic,0.83,0.73,0.76
1,Cardboard,0.76,0.71,0.78
2,Paper,0.93,0.76,0.83
8,Plastic cup,0.74,0.89,0.87
4,Glass bottle,0.87,0.76,0.90
0,Aluminum can,0.93,0.91,0.94
